In [1]:
import csv
import matplotlib.pyplot as plt
from pathlib import Path

def moving_average(values, window_fraction=0.25):
    window_size = max(1, int(window_fraction * len(values)))
    smoothed = []

    for i in range(len(values)):
        start_idx = max(0, i - window_size + 1)
        window_mean = sum(values[start_idx : i + 1]) / (i - start_idx + 1)
        smoothed.append(window_mean)

    return smoothed


def plot_grpo_metrics(csv_path, columns, save_as=None):
    data = {name: {"steps": [], "values": []} for name in columns}

    with Path(csv_path).open(newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if not row or not row.get("step"):
                continue

            step = int(row["step"])

            for name in columns:
                value_str = row.get(name)
                if value_str:
                    data[name]["steps"].append(step)
                    data[name]["values"].append(float(value_str))

    fig, axes = plt.subplots(2, 2, sharex=True, figsize=(6, 4))
    axes = axes.ravel()

    for i, name in enumerate(columns):
        steps = data[name]["steps"]
        values = data[name]["values"]

        if not values:
            fig.delaxes(axes[i])
            continue

        if name == "eval_acc":
            axes[i].bar(steps, values, width=20)
        else:
            axes[i].plot(steps, values, alpha=0.4)
            axes[i].plot(steps, moving_average(values))

        axes[i].set_ylabel(name)

    for j in (2, 3):
        if axes[j] in fig.axes:
            axes[j].set_xlabel("Step")

    plt.tight_layout()
    if save_as is not None:
        plt.savefig(save_as)
    plt.show()

plot_grpo_metrics(
    "rlvr_grpo.csv",
    columns=["loss", "reward_avg", "avg_response_len", "eval_acc"],
)

FileNotFoundError: [Errno 2] No such file or directory: 'rlvr_grpo.csv'

In [ ]:
import torch

def compute_advanatage_stats(rewards_list):
    rewards = torch.tensor(rewards_list)
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)
    
    adv_avg = advantages.mean().item()
    adv_std = advantages.std().item()
    
    return advantages, adv_avg, adv_std

In [ ]:
adv, adv_avg, adv_std = compute_advanatage_stats([1., 1., 0., 0.])
print(f"Advantages: {adv}")
print(f"Advantages mean = {adv_avg:.4f}, std = {adv_std:.4f}")

Advantages: tensor([ 0.8659,  0.8659, -0.8659, -0.8659])
Advantages mean = 0.0000, std = 0.9998


The standard deviation has more information here 
- std ≈ 1 means we have a good scale of the gradient signal for stable updates 
- std ≪ 1 means vanishing signal (often happens when rewards collapse) 
- std ≫ 1 means overly spiky updates that can destabilize training

In [ ]:
adv, adv_avg, adv_std = compute_advanatage_stats([0., 0., 0., 0.])
print(f"Advantages: {adv}")
print(f"Advantages mean = {adv_avg:.4f}, std = {adv_std:.4f}")

Advantages: tensor([0., 0., 0., 0.])
Advantages mean = 0.0000, std = 0.0000


In [ ]:
adv, adv_avg, adv_std = compute_advanatage_stats([1., 1., 1., 1.])
print(f"Advantages: {adv}")
print(f"Advantages mean = {adv_avg:.4f}, std = {adv_std:.4f}")

Advantages: tensor([0., 0., 0., 0.])
Advantages mean = 0.0000, std = 0.0000


In [ ]:
logits = torch.tensor([0.6667, -2.0000,  1.3333, -0.0000, -0.6667,  2.0000, -1.3333])

In [ ]:
logprobs = torch.log_softmax(logits, dim=-1)
print("All token logprobs:", logprobs)

selected_token = torch.argmax(logprobs)

selected = logprobs[selected_token]
print("Selected token ID:", selected_token)
print("Selected token logprob", selected)

All token logprobs: tensor([-2.0442, -4.7109, -1.3776, -2.7109, -3.3776, -0.7109, -4.0442])
Selected token ID: tensor(5)
Selected token logprob tensor(-0.7109)


entropy is calculated by multiplying the probability values by the log-probability values, then we sum these resulting products 
- Very low entropy (≈ 0–0.5) means one token dominates the distribution, where the model is highly confident and behaves almost deterministically 
- Moderate entropy (≈ 1–2) means that the probability mass is shared among a few more tokens, which is typical during stable training 
- High entropy (≫ 2, approaching log(vocabulary_size); here: log(7) = 1.9459), the probability mass is spread across many tokens, and the model is highly uncertain and behaves close to random

In [ ]:
probs = torch.softmax(logits, dim=-1)
logprobs = torch.log_softmax(logits, dim=-1)

entropy = torch.sum(-(probs * logprobs))

print("Probs:", probs)
print("Entropy:", entropy)

Probs: tensor([0.1295, 0.0090, 0.2522, 0.0665, 0.0341, 0.4912, 0.0175])
Entropy: tensor(1.3700)


In [ ]:
print("Probs", torch.exp(logprobs))

Probs tensor([0.1295, 0.0090, 0.2522, 0.0665, 0.0341, 0.4912, 0.0175])


In [ ]:
def sequence_logprob_and_entropy(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)
    
    targets = token_ids[1:]
    selected = logprobs[:-1].gather(1, token_ids[1:].unsqueeze(-1)).squeeze(-1)
    
    selected_answer_logprobs = selected[prompt_len - 1:]
    logp_all_steps = torch.sum(selected_answer_logprobs)
    
    all_answer_logprobs = logprobs[:-1][prompt_len - 1:]
    if all_answer_logprobs.numel() == 0:
        entropy_all_steps = logp_all_steps.new_tensor(0.0) # end if eos
    else:
        all_answer_probs = torch.exp(all_answer_logprobs) # logprob to prob
        plogp = all_answer_probs * all_answer_logprobs # elementwise multiplication
        step_entropy = -torch.sum(plogp, dim=-1) # sum
        entropy_all_steps = torch.mean(step_entropy) # mean entropy
    
    return logp_all_steps, entropy_all_steps

![image.png](https://cdn.hackclub.com/01a139b0-e2a1-779f-b913-62bf10190897/download.png)

- The average advantage is and stays zero, which is exactly what we expect with GRPO-style normalization
    - Advantages are computed relative to the group mean, so by construction, they should sum to zero
    - This metric is mostly a sanity check in GRPO, and if it drifted away from zero, that would point to a bug or other normalization bias issue
- The advantage std is the more informative signal
    - Early on, we see a relatively high variance, which means the model produces diverse rollouts
    - Later on, the advantage std slowly declines and stabilizes, which means rollouts become more similar in quality
    - But the bottom line is that a nonzero (and stable) advantage std means there is still a usable learning signal
- Next, let’s look at the entropy
    - Early in training, the entropy is relatively flat and low (this indicates that the LLM is fairly deterministic)
    - In practice, increasing the sampling temperature would increase the output diversity during rollout, although it does not change the underlying LLM entropy
    - Later in training (after ~step 200), the entropy increases substantially, which means the next-token distribution is getting broader and the model is becoming more stochastic
    - Very low entropy can be a sign of collapse, where the model repeatedly produces the same or very similar outputs
    - In this run, however, considering entropy together with the reward average and the non-vanishing advantage standard deviation suggests an overall healthy exploration rather than collapse


In [ ]:
rewards = torch.tensor([1., 1., 0., 0.])
logprobs = torch.tensor([-7.9243, -20.1546, -16.6130, -23.3677])

advantages = torch.tensor(rewards - rewards.mean()) / (rewards.std() + 1e-4)

pg_loss = -(advantages.detach() * logprobs).mean()
print("Policy gradient loss:", pg_loss)

Policy gradient loss: tensor(-2.5764)


C:\Users\hridd\AppData\Local\Temp\ipykernel_4344\46840007.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  advantages = torch.tensor(rewards - rewards.mean()) / (rewards.std() + 1e-4)


In [ ]:
new_logps = logprobs

old_logps = torch.tensor([
    -10.9243,   # -7.9243
    -20.3546,   # -20.1546
    -14.6130,   # -16.6130
    -23.3677,   # -23.3677
])

log_ratio = new_logps - old_logps
ratio = torch.exp(log_ratio)
clip_eps = 10.0
clipped_ratio = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps)

print("Ratio:", ratio)
print("Clipped ratio:", clipped_ratio)

Ratio: tensor([20.0855,  1.2214,  0.1353,  1.0000])
Clipped ratio: tensor([11.0000,  1.2214,  0.1353,  1.0000])


In [ ]:
adv = advantages.detach() 

unclipped = ratio * adv
clipped = clipped_ratio * adv

obj = torch.minimum(unclipped, clipped)

clipped_pg_loss = -torch.mean(obj)
policy_ratio = torch.mean(ratio)

print("Clipped policy gradient loss:", clipped_pg_loss)
print("Policy ratio:", policy_ratio)

Clipped policy gradient loss: tensor(-2.3998)
Policy ratio: tensor(5.6106)


![i](https://cdn.hackclub.com/01a13a13-002c-7f4b-9a68-fc523805e2c5/download%20(1).png)

![](https://cdn.hackclub.com/01a13a13-00df-7f1f-b4ba-c4dafc6d1e82/download%20(2).png)

In [ ]:
import copy

kl_coeff = 0.0
if kl_coeff:
    ref_model = copy.deepcopy(model).to(device)
    ref_model.eval()
    for p in ref_model.parameters():
        p.requires_grad = False
else:
    ref_model = None
    
def compute_grpo_loss_with_kl(
    model,
    ref_model,
    tokenizer,
    example,
    device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
    kl_coeff=0.02,
):
    assert num_rollouts >= 2
    roll_logps, roll_ref_logps, roll_rewards, samples = [], [], [], []
    prompt = render_prompt(example["problem"])
    
    was_training = model.training
    model.eval()
    
    for _ in range(num_rollouts):
        token_ids, prompt_len, text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )
        
        if kl_coeff:
            with torch.no_grad():
                ref_logp = sequence_logprop(ref_model, token_ids, prompt_len)
        else:
            ref_logp = None
        
        reward = reward_rlvr(text, example["answer"])
        logp = sequence_logprob(model, token_ids, prompt_len)
        
        roll_logps.append(logp)
        roll_rewards.append(reward)
        
        if kl_coeff:
            roll_ref_logps.append(ref_logp)
        
        samples.append({"text": text, "reward": reward, "gen_len": token_ids.numel() - prompt_len,})
        
    if was_training:
        model.train()
        
    rewards = torch.tensor(roll_rewards, device=device)
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)
    logps = torch.stack(roll_logps)
    
    if kl_coeff:
        ref_logp = torch.stack(roll_ref_logps).detach()
    
    pg_loss = -(advantages.detach() * logps).mean()
    
    if kl_coeff:
        kl_loss = kl_coeff * torch.mean(logps - ref_logps)
    else:
        kl_loss = torch.tensor(0.0, device=logps.device)
        
    loss = pg_loss + kl_loss
    
    return {
        "loss": loss.item(),
        "pg_loss": pg_loss.item(),
        "rewards": roll_rewards,
        "advantages": advantages.detach().cpu().tolist(),
        "samples": samples,
        "loss_tensor": loss,
    }

![](https://cdn.hackclub.com/01a13a1d-361d-7df7-8dbf-190968748623/download%20(3).png)

![](https://cdn.hackclub.com/01a13a1d-b15f-7d12-9440-8d8ac8b0eb84/download%20(4).png)

this is bad

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd "drive/MyDrive/reasoning"

/content/drive/MyDrive/reasoning


In [5]:
from helpers.ch03 import load_model_and_tokenizer
import torch 

device = torch.device("cpu")
model, tokenizer_base = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False,
) 

✓ qwen3/qwen3-0.6B-base.pth already up-to-date


In [6]:
print(tokenizer_base.encode("<think>"))

[13708, 766, 29]


In [9]:
for i in [13708, 766, 29]:
    print(tokenizer_base.decode([i]))

<th
ink
>


In [10]:
tokenizer_base._tok.add_special_tokens(["<tool_response>", "</tool_response>", "<think>", "</think>"])

4

In [12]:
print(tokenizer_base.encode("<think>"))

[151667]


In [13]:
print(tokenizer_base.encode("</think>"))

[151668]


In [15]:
print("Vocabulary size:", model.tok_emb.weight.shape[0])

Vocabulary size: 151936


In [16]:
from helpers.qwen3 import Qwen3Tokenizer
from helpers.qwen3 import download_qwen3_small

download_qwen3_small(kind="reasoning", tokenizer_only=True, out_dir="qwen3")
tokenizer_path = Path("qwen3") / "tokenizer-reasoning.json"
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

tokenizer-reasoning.json: 100% (10 MiB / 10 MiB)


In [17]:
print(tokenizer_base.encode("<think>"))

[151667]


In [18]:
print(tokenizer_base.encode("</think>"))

[151668]


In [24]:
def reward_format(token_ids, prompt_len, start_think_id=151667, end_think_id=151668):
    try:
        gen = token_ids[prompt_len:].tolist()
        return float(gen.index(start_think_id) < gen.index(end_think_id))
    except ValueError:
        return 0.0

In [25]:
prompt = "Calculate ..."
rollout = "Let's ... <think> ... </think> ..."
token_ids = tokenizer.encode(prompt + rollout)

reward_format(
    token_ids=torch.tensor(token_ids),
    prompt_len=len(tokenizer.encode(prompt))
)

1.0

In [27]:
def compute_grpo_loss_plus_format_reward(
    model,
    tokenizer,
    example,
    device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
    format_reward_weight = 1.0,
):
    assert num_rollouts >= 2
    roll_logps, roll_rewards, samples = [], [], [],
    prompt = render_prompt(example["problem"])
    
    was_training = model.training
    model.eval()
    
    for _ in range(num_rollouts):
        token_ids, prompt_len, text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )
        
        reward = reward_rlvr(text, example["answer"])
        logp = sequence_logprob(model, token_ids, prompt_len)
        format_reward = reward_format(token_ids, prompt_len)
        reward = rlvr + format_reward_weight * format_reward
        
        roll_logps.append(logp)
        roll_rewards.append(reward)
        
        samples.append({"text": text, "reward": reward, "gen_len": token_ids.numel() - prompt_len,})
        
    if was_training:
        model.train()
        
    rewards = torch.tensor(roll_rewards, device=device)
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)
    logps = torch.stack(roll_logps)
    
    pg_loss = -(advantages.detach() * logps).mean()
    loss = pg_loss
    
    return {
        "loss": loss.item(),
        "pg_loss": pg_loss.item(),
        "rewards": roll_rewards,
        "advantages": advantages.detach().cpu().tolist(),
        "samples": samples,
        "loss_tensor": loss,
    }

In [ ]:
def render_prompt_with_think_tokens(prompt):
    template = (
        "You are a helpful math assistant.\n"
        "When solving the problem, first write your reasoning inside <think> and </think> tags.\n"
        "Then write the final result on a new line in the exact format:\n"
        "\\boxed{ANSWER}\n\n"
        f"Question:\n{prompt}\n\nAnswer:"
    )
    return template

![](https://cdn.hackclub.com/01a13a2e-10ff-7fde-911b-aea03206bacf/download%20(5).png)

![](https://cdn.hackclub.com/01a13a2e-11d4-7b4c-98c1-37fbdc349b92/download%20(6).png)